In [11]:
# importa as bilbiotecas

import requests as rq
import urllib3
import pandas as pd

In [12]:
#faz o request
# Evita warning de certificado SSL (para testes locais)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

url = 'https://api-comexstat.mdic.gov.br/general'

headers = {
    'Accept': 'application/json',
    'Content-Type': 'application/json'
}

body = {
   "flow": "import",
   "monthDetail": True,
   "period":{
       "from": "2025-01",
       "to": "2025-12"
   },
   "filters":[
       {
           "filter":"heading",
           "values":[6401,6402,6403,6404,6405]
       }
   ],
   "details":[
       "country",
       "state",
       "ncm",
       "heading"
   ],
   "metrics":[
       "metricFOB",
       "metricKG",
       "metricStatistic"
   ]

}

response = rq.post(url, headers=headers, json=body, verify=False)

print("Status Code:", response.status_code)
print(response.text)



Status Code: 200
{"data":{"list":[{"coNcm":"64041100","year":"2025","monthNumber":"01","country":"Vietn\u00e3","state":"Minas Gerais","ncm":"Cal\u00e7ados para esportes, etc, de mat\u00e9rias t\u00eaxteis, sola borracha\/pl\u00e1stico","headingCode":"6404","heading":"Cal\u00e7ado com sola exterior de borracha, pl\u00e1stico, couro natural ou reconstitu\u00eddo e parte superior de mat\u00e9rias t\u00eaxteis","metricFOB":"7226671","metricKG":"200820","metricStatistic":"271790"},{"coNcm":"64041100","year":"2025","monthNumber":"03","country":"Indon\u00e9sia","state":"Minas Gerais","ncm":"Cal\u00e7ados para esportes, etc, de mat\u00e9rias t\u00eaxteis, sola borracha\/pl\u00e1stico","headingCode":"6404","heading":"Cal\u00e7ado com sola exterior de borracha, pl\u00e1stico, couro natural ou reconstitu\u00eddo e parte superior de mat\u00e9rias t\u00eaxteis","metricFOB":"7162317","metricKG":"270591","metricStatistic":"377580"},{"coNcm":"64041100","year":"2025","monthNumber":"02","country":"Vietn

In [13]:
#transforma o retorno em json
json_dados = response.json()
print(json_dados)

{'data': {'list': [{'coNcm': '64041100', 'year': '2025', 'monthNumber': '01', 'country': 'Vietnã', 'state': 'Minas Gerais', 'ncm': 'Calçados para esportes, etc, de matérias têxteis, sola borracha/plástico', 'headingCode': '6404', 'heading': 'Calçado com sola exterior de borracha, plástico, couro natural ou reconstituído e parte superior de matérias têxteis', 'metricFOB': '7226671', 'metricKG': '200820', 'metricStatistic': '271790'}, {'coNcm': '64041100', 'year': '2025', 'monthNumber': '03', 'country': 'Indonésia', 'state': 'Minas Gerais', 'ncm': 'Calçados para esportes, etc, de matérias têxteis, sola borracha/plástico', 'headingCode': '6404', 'heading': 'Calçado com sola exterior de borracha, plástico, couro natural ou reconstituído e parte superior de matérias têxteis', 'metricFOB': '7162317', 'metricKG': '270591', 'metricStatistic': '377580'}, {'coNcm': '64041100', 'year': '2025', 'monthNumber': '02', 'country': 'Vietnã', 'state': 'Minas Gerais', 'ncm': 'Calçados para esportes, etc, 

In [14]:
#cria dataframe
normaliza_dados = pd.json_normalize(json_dados['data']['list'])
data_frame = pd.DataFrame(normaliza_dados)



In [15]:
#ajusta o tipo dos dados
data_frame = data_frame.astype({
    'year': int,
    'monthNumber': int,
    'coNcm': int,
    'metricFOB': float,
    'metricKG': float,
    'metricStatistic': float,
    'headingCode': int
})

In [16]:
#ordena os dados
data_frame = data_frame.sort_values(by='monthNumber', ascending=True)

In [17]:
#cria coluna dia
data_frame['dia'] = 1

In [18]:
#cria coluna data
data_frame['data'] = (data_frame['dia'].astype(str).str.zfill(2)+'/'+data_frame['monthNumber'].astype(str).str.zfill(2)+'/'+data_frame['year'].astype(str))

In [19]:
#ordena colunas
data_frame = data_frame[['year','monthNumber','dia','data','headingCode','heading','coNcm','ncm','country','state','metricFOB','metricKG','metricStatistic']]

In [20]:
#salva dados em excel
data_frame.to_excel('./dados_importacao.xlsx', header=True, index=False, sheet_name='importacao')